# Practical 8 — Open-Source GenAI with Hugging Face

**Enrollment:** BT24S05F007

## Aim
Load an open-source language model from Hugging Face, inspect it, run inference, perform a short fine-tuning pass on a local corpus, then generate text again to show the deployment-style workflow.

## Model
`distilgpt2` — a compact GPT-2 variant that fits a student laptop and still demonstrates tokenization, causal language modeling, and fine-tuning.


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"
torch.manual_seed(7)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
print("Loaded:", MODEL_NAME)
print("Architecture:", model.config.architectures)
print("Layers:", model.config.n_layer, "| Embed dim:", model.config.n_embd)
print("Parameters:", sum(p.numel() for p in model.parameters()))


/private/tmp/genai-lab/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 17:51:53.853000 10451 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 21174.91it/s]

Loaded: distilgpt2
Architecture: ['GPT2LMHeadModel']
Layers: 6 | Embed dim: 768
Parameters: 81912576


## Baseline generation before fine-tuning


In [2]:
def continue_text(net, prompt, new_tokens=40):
    net.eval()
    encoded = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = net.generate(
            **encoded,
            max_new_tokens=new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=40,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

base_prompt = "In a Generative AI laboratory, students learn that"
print("Before fine-tuning:\n", continue_text(model, base_prompt))


Before fine-tuning:
 In a Generative AI laboratory, students learn that information is not just "random" but "unrandom" by creating complex neural networks of neurons.




The team developed a neural network of neurons to encode the information of the neural


## Tiny fine-tuning corpus
A few course-specific sentences are enough to show the training loop. This is not production fine-tuning; it is a controlled classroom pass on Hugging Face weights.


In [3]:
corpus = [
    "Retrieval Augmented Generation grounds answers in retrieved notes.",
    "A GAN trains a generator against a discriminator on real and fake samples.",
    "Diffusion models denoise random noise step by step until a sample appears.",
    "Neural style transfer keeps content layout while copying style statistics.",
    "Hugging Face provides open tokenizers and pretrained generative models.",
    "Prompt engineering sets the role, task, format, and limits for a model.",
]

encodings = tokenizer(
    corpus,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64,
)
input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]
labels = input_ids.clone()
labels[attention_mask == 0] = -100

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()
losses = []
for epoch in range(1, 8):
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    losses.append(float(loss.item()))
    print(f"epoch {epoch}: loss={loss.item():.4f}")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch 1: loss=7.1317
epoch 2: loss=6.5066


epoch 3: loss=6.1456
epoch 4: loss=5.6553


epoch 5: loss=5.4414
epoch 6: loss=4.8796


epoch 7: loss=4.5874


## Generation after fine-tuning and a simple local ‘deploy’ save


In [4]:
print("After fine-tuning:\n", continue_text(model, base_prompt))
print()
print(continue_text(model, "Diffusion models denoise"))

save_dir = "hf_distilgpt2_lab_checkpoint"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("\nSaved local deployment folder:", save_dir)

reloaded = AutoModelForCausalLM.from_pretrained(save_dir)
reloaded_tok = AutoTokenizer.from_pretrained(save_dir)
print("Reload check vocab size:", reloaded_tok.vocab_size)
print("Final training losses:", [round(v, 4) for v in losses])


After fine-tuning:
 In a Generative AI laboratory, students learn that a random face is a random face that has a random face in its name. The random face can also be generated in a generator. A random face can then be generated in a generator. For example



Diffusion models denoise the sample size when it comes to quality. To determine sample sizes, we used an order of magnitude of a set of samples. We chose to use a random sampling order to avoid bias. A sample


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.70it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.68it/s]


Saved local deployment folder: hf_distilgpt2_lab_checkpoint


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 21128.59it/s]

Reload check vocab size: 50257
Final training losses: [7.1317, 6.5066, 6.1456, 5.6553, 5.4414, 4.8796, 4.5874]


## Conclusion
The Hugging Face workflow used here is: load an open model → run inference → fine-tune briefly on a domain corpus → save a local checkpoint for reuse. That is the same pipeline used for larger open models, scaled down for a laboratory machine.


## Run results

Outputs captured from the executed notebook.

```
Loaded: distilgpt2
Architecture: ['GPT2LMHeadModel']
Layers: 6 | Embed dim: 768
Parameters: 81912576
```

```
Before fine-tuning:
 In a Generative AI laboratory, students learn that information is not just "random" but "unrandom" by creating complex neural networks of neurons.




The team developed a neural network of neurons to encode the information of the neural
```

```
epoch 1: loss=7.1317
epoch 2: loss=6.5066
epoch 3: loss=6.1456
epoch 4: loss=5.6553
epoch 5: loss=5.4414
epoch 6: loss=4.8796
epoch 7: loss=4.5874
```

```
After fine-tuning:
 In a Generative AI laboratory, students learn that a random face is a random face that has a random face in its name. The random face can also be generated in a generator. A random face can then be generated in a generator. For example

Diffusion models denoise the sample size when it comes to quality. To determine sample sizes, we used an order of magnitude of a set of samples. We chose to use a random sampling order to avoid bias. A sample

Saved local deployment folder: hf_distilgpt2_lab_checkpoint
Reload check vocab size: 50257
Final training losses: [7.1317, 6.5066, 6.1456, 5.6553, 5.4414, 4.8796, 4.5874]
```
